# GNN Training and Embedding Export

This notebook runs the full GNN pipeline:
1. Loads the knowledge graph pickle
2. Builds a heterogeneous PyG graph
3. Trains the HeteroGNN with link prediction
4. Exports the trained function embeddings to pickle (and optionally to Neo4j)

In [12]:
import os
import sys
import torch

sys.path.append(os.path.abspath('package'))

from package.gnn import (
    load_knowledge_graph,
    build_hetero_data,
    split_edges,
    HeteroGNN,
    LinkPredictor,
)
from package.gnn.train import train_step, evaluate
from package.gnn.export import compute_embeddings, export_to_pickle, export_to_neo4j

## 1. Configuration

Set hyperparameters here. Update `KG_PKL` to point to your saved pickle file.

In [13]:
KG_PKL = 'graph_v10_nosubgraph.pkl'
OUT_DIR = 'gnn_out'
ENCODER = 'sentence-transformers/all-MiniLM-L6-v2'

EPOCHS = 100
HIDDEN = 256
OUT_DIM = 256
LAYERS = 2
LR = 1e-3
WEIGHT_DECAY = 1e-5

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
os.makedirs(OUT_DIR, exist_ok=True)
print(f'Device: {DEVICE}')
print(f'Output dir: {OUT_DIR}')

Device: cuda
Output dir: gnn_out


## 2. Load Knowledge Graph

In [14]:
print(f'Loading {KG_PKL}...')
kg = load_knowledge_graph(KG_PKL)

print('\nNode/edge counts:')
for key in kg.keys():
    df = kg[key]
    if hasattr(df, '__len__') and (key.endswith('_nodes') or key.endswith('_edges')):
        print(f'  {key:40s}: {len(df):>10,}')

Loading graph_v10_nosubgraph.pkl...

Node/edge counts:
  function_nodes                          :     10,891
  function_edges                          :     23,666
  subgraph_nodes                          :  2,065,852
  subgraph_edges                          :  2,054,961
  subgraph_function_edges                 :  2,065,852
  function_subgraph_edges                 :  2,065,852
  import_nodes                            :      2,561
  class_nodes                             :        852
  class_function_edges                    :      3,411
  class_class_edges                       :        951
  file_nodes                              :        754
  file_edges                              :        753
  file_function_edges                     :     10,891
  file_class_edges                        :        906
  file_import_edges                       :      9,690
  config_nodes                            :         21
  file_config_edges                       :         21
  import_f

## 3. Build HeteroData

Computes text-based initial embeddings (SentenceTransformer) for each node type,
loads the edge types defined in EDGE_SPECS, and adds reverse edges for bidirectional message passing.

In [15]:
data, id_maps = build_hetero_data(kg, encoder_model=ENCODER, device=DEVICE)
print(data)

Batches: 100%|██████████| 79/79 [00:08<00:00,  9.64it/s]


HeteroData(
  function={ x=[10891, 384] },
  import={ x=[2561, 384] },
  cluster={ x=[50, 384] },
  class={ x=[852, 384] },
  file={ x=[754, 384] },
  dfg={ x=[20061, 384] },
  (function, calls, function)={ edge_index=[2, 23666] },
  (cluster, groups, function)={ edge_index=[2, 10741] },
  (class, owns, function)={ edge_index=[2, 3411] },
  (file, contains, function)={ edge_index=[2, 10891] },
  (dfg, flows_in, function)={ edge_index=[2, 20061] },
  (class, extends, class)={ edge_index=[2, 951] },
  (function, rev_calls, function)={ edge_index=[2, 23666] },
  (function, rev_groups, cluster)={ edge_index=[2, 10741] },
  (function, rev_owns, class)={ edge_index=[2, 3411] },
  (function, rev_contains, file)={ edge_index=[2, 10891] },
  (function, rev_flows_in, dfg)={ edge_index=[2, 20061] },
  (class, rev_extends, class)={ edge_index=[2, 951] }
)


## 4. Train/Val/Test Split on `function calls function` Edges

In [16]:
SUPERVISION_EDGE = ('function', 'calls', 'function')

train_data, val_data, test_data = split_edges(data, SUPERVISION_EDGE)
train_data = train_data.to(DEVICE)
val_data = val_data.to(DEVICE)
test_data = test_data.to(DEVICE)

for name, d in [('train', train_data), ('val', val_data), ('test', test_data)]:
    pos = (d[SUPERVISION_EDGE].edge_label == 1).sum().item()
    neg = (d[SUPERVISION_EDGE].edge_label == 0).sum().item()
    print(f'{name}: {pos} positive, {neg} negative supervision edges')

train: 20117 positive, 20117 negative supervision edges
val: 1183 positive, 1183 negative supervision edges
test: 2366 positive, 2366 negative supervision edges


## 5. Model Initialization

In [17]:
in_channels_dict = {nt: data[nt].x.size(-1) for nt in data.node_types}

model = HeteroGNN(
    node_types=list(data.node_types),
    edge_types=list(data.edge_types),
    in_channels_dict=in_channels_dict,
    hidden_channels=HIDDEN,
    out_channels=OUT_DIM,
    num_layers=LAYERS,
).to(DEVICE)

predictor = LinkPredictor().to(DEVICE)

optimizer = torch.optim.Adam(
    list(model.parameters()) + list(predictor.parameters()),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

n_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {n_params:,}')
print(f'Node types: {list(data.node_types)}')
print(f'Edge types: {len(data.edge_types)}')

Model parameters: 4,137,984
Node types: ['function', 'import', 'cluster', 'class', 'file', 'dfg']
Edge types: 12


## 6. Training Loop

Validation after every epoch. Best val AUC checkpoint is saved.

In [18]:
losses, val_aucs = [], []
best_val = 0.0
checkpoint_path = os.path.join(OUT_DIR, 'best_model.pt')

for epoch in range(1, EPOCHS + 1):
    loss = train_step(model, predictor, optimizer, train_data, SUPERVISION_EDGE)
    val_auc = evaluate(model, predictor, val_data, SUPERVISION_EDGE)
    losses.append(loss)
    val_aucs.append(val_auc)

    marker = ''
    if val_auc > best_val:
        best_val = val_auc
        marker = ' *'
        torch.save({
            'model_state': model.state_dict(),
            'predictor_state': predictor.state_dict(),
            'id_maps': id_maps,
            'config': {
                'node_types': list(data.node_types),
                'edge_types': list(data.edge_types),
                'in_channels_dict': in_channels_dict,
                'hidden': HIDDEN,
                'out_dim': OUT_DIM,
                'layers': LAYERS,
                'encoder_model': ENCODER,
            },
        }, checkpoint_path)

    if epoch == 1 or epoch % 5 == 0 or epoch == EPOCHS:
        print(f'Epoch {epoch:03d} | loss={loss:.4f} | val_auc={val_auc:.4f}{marker}')

test_auc = evaluate(model, predictor, test_data, SUPERVISION_EDGE)
print(f'\nBest val AUC: {best_val:.4f} | Test AUC: {test_auc:.4f}')
print(f'Checkpoint saved to: {checkpoint_path}')

Epoch 001 | loss=0.7049 | val_auc=0.7671 *
Epoch 005 | loss=0.6901 | val_auc=0.8512
Epoch 010 | loss=0.6304 | val_auc=0.8141
Epoch 015 | loss=0.5882 | val_auc=0.7997
Epoch 020 | loss=0.5759 | val_auc=0.8351
Epoch 025 | loss=0.5574 | val_auc=0.8166
Epoch 030 | loss=0.5529 | val_auc=0.8135
Epoch 035 | loss=0.5463 | val_auc=0.8237
Epoch 040 | loss=0.5388 | val_auc=0.8168
Epoch 045 | loss=0.5338 | val_auc=0.8202
Epoch 050 | loss=0.5294 | val_auc=0.8233
Epoch 055 | loss=0.5249 | val_auc=0.8241
Epoch 060 | loss=0.5205 | val_auc=0.8278
Epoch 065 | loss=0.5174 | val_auc=0.8255
Epoch 070 | loss=0.5104 | val_auc=0.8319
Epoch 075 | loss=0.5063 | val_auc=0.8313
Epoch 080 | loss=0.5024 | val_auc=0.8349
Epoch 085 | loss=0.4975 | val_auc=0.8319
Epoch 090 | loss=0.4943 | val_auc=0.8273
Epoch 095 | loss=0.4907 | val_auc=0.8375
Epoch 100 | loss=0.4885 | val_auc=0.8394

Best val AUC: 0.8564 | Test AUC: 0.8244
Checkpoint saved to: gnn_out\best_model.pt


In [ ]:
import json

history = {
    'losses': losses,
    'val_aucs': val_aucs,
    'best_val': best_val,
    'test_auc': float(test_auc),
}
history_path = os.path.join(OUT_DIR, 'training_history.json')
with open(history_path, 'w') as f:
    json.dump(history, f)
print(f'Training history saved to: {history_path}')

## 7. Training Curve

In [ ]:
import matplotlib.pyplot as plt

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
axs[0].plot(losses)
axs[0].set_xlabel('Epoch')
axs[0].set_ylabel('BCE loss')
axs[0].set_title('Training loss')
axs[0].grid(True, alpha=0.3)

axs[1].plot(val_aucs)
axs[1].axhline(best_val, color='red', linestyle='--', alpha=0.5, label=f'best={best_val:.3f}')
axs[1].set_xlabel('Epoch')
axs[1].set_ylabel('AUC')
axs[1].set_title('Validation AUC')
axs[1].legend()
axs[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Compute Embeddings and Save to Pickle

Runs a single forward pass with the best checkpoint. Output: `{function_id: [256 floats]}` dict.

In [ ]:
embs = compute_embeddings(checkpoint_path, KG_PKL, device=DEVICE)
print(f'Computed {len(embs):,} function embeddings.')

first_id = next(iter(embs))
print(f'Sample (function {first_id}): dim={len(embs[first_id])}, first 5 values={embs[first_id][:5]}')

pkl_path = os.path.join(OUT_DIR, 'gnn_embeddings.pkl')
export_to_pickle(embs, pkl_path)
print(f'Saved to {pkl_path}')

## 9. (Optional) Write Embeddings to Neo4j

Every FUNCTION node gets a `gnn_embedding` property. Only run if Neo4j is running.